# 09B · Does modeling RNA diffusion improve cell annotation in one mouse?

This pilot fits **raw-count PCA, standard scVI, and spatial ResolVI** on the same observed cell segments and measured genes, then runs the same Leiden settings on each representation. The default is a contiguous 12,000-cell core from Ada-14R with a 100 µm halo. It includes every cell type in the region. Select **full** in the configuration for every eligible cell in one mouse.

Start in the Adagrasib project environment with scvi-tools, Pyro, scikit-learn, scanpy (igraph Leiden support), python-igraph, anndata and a GPU-enabled PyTorch installation. This pilot uses the scvi-tools 1.3.3 / 1.5.0.post1 ResolVI API. After updating the notebook and its helper modules, restart the GPU-enabled kernel and run all cells. Point `OUTPUT_ROOT` at the existing `09_pilot_results` directory to reuse the prepared sample. Both sample choices and earlier cluster labels are exploratory. [ResolVI model guide](https://docs.scvi-tools.org/en/1.3.3/user_guide/models/resolvi.html)

## Configure one sample and the training region

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
from input09 import prepare_sample
from review09 import show_review_zip
from resolvi09 import run

SAMPLE = "Ada-14R"
PROJECT_ROOT = None  # Set the same project or CT26 run root used by 09A if needed.
SAMPLE_H5AD = None   # Optional one-mouse, all-cell H5AD with original RNA and micron centroids.
OUTPUT_ROOT = Path("09_pilot_results").resolve()
MODE = "tile"  # "tile" or "full"
FIT = dict(target_core_cells=12_000, halo_um=100, section=None, center_um=None,
           n_features=1_200, n_neighbors=8, batch_size=64, epochs=40, latent_dim=16,
           resolution=0.6, min_model_counts=5, run_scvi=True, accelerator="gpu",
           block_um=200, marker_cell_cap=30_000, seed=17)
# Optional label overrides are regexes on existing annotation names:
# FIT["type_patterns"] = {"macrophage": r"macroph|monocyt"}

## Load the all-cell count source

The adapter shared with 09A accepts the same project root and cached sample file. Model feature selection uses up to 6,000 cells. Training and Leiden use all eligible segments from the configured footprint. The selected-gene library must contain at least five observed molecules by default; the receipt reports exclusions. CD4, CD8α and CD8β transcripts are withheld from feature selection and from all three representations.

In [ ]:
sample_path = prepare_sample(SAMPLE, OUTPUT_ROOT, project_root=PROJECT_ROOT, sample_h5ad=SAMPLE_H5AD)
print("Pilot count source:", sample_path)

## Fit and compare

The tile center defaults to a cell near the median coordinates of the largest section. Set `center_um=(x,y)` to use a prespecified region. Spatial neighbors are exact, exclude self, and stay within the same section. The halo supplies possible neighbors for core cells. Near an artificial tile edge only cells whose neighbor distances fit inside the tile are evaluated. The full-sample setting preserves all available regions.

Before reading the count matrix for the fit, 09B runs a small calculation on the first GPU visible to this kernel. The run records the outcome in `runtime.json`. If GPU calculations work but the peak-memory counter is unavailable, training continues and the peak is reported as unavailable. After the model-gene count filter, 09B uses the number of eligible core and halo cells to choose one training batch size for scVI and ResolVI. The requested `batch_size` is an upper bound. The notebook adjusts it when needed so the final training batch contains multiple cells and every eligible cell participates in each training epoch. The fit summary reports the effective batch size. The notebook uses sparse counts and minibatches on one GPU. It decodes marker panels for up to 30,000 core cells and samples the mixture for up to 1,000. Watch the printed footprint and GPU usage on your first run. A reasonable full-sample estimate starts with the tile's fit minutes and the ratio of eligible cell counts; graph construction and posterior inference scale differently. A tile cannot characterize section-wide heterogeneity.

In [ ]:
try:
    out, review = run(sample_path, OUTPUT_ROOT, mode=MODE, **FIT)
except Exception as exc:
    diagnostic_zip = getattr(exc, "diagnostics_zip", None)
    if diagnostic_zip is not None:
        show_review_zip(diagnostic_zip)
    raise
print("Local results:", out)
show_review_zip(review)
display(pd.Series(json.loads((out / "complete.json").read_text())))

## Examine the biology on identical cells

The AUC assesses raw exclusive CD4/CD8 anchors among pre-annotated T cells with observed pan-T support. Direct coreceptors never enter the model. The classifier holds out spatial blocks within the sample; `n_scored` and `n_blocks` indicate coverage. These are within-mouse RNA comparisons. The broad annotation agreement (AMI) measures concordance with existing labels.

The specificity table compares mean marker signal in pan-T-positive annotated T cells with the appropriate source population. Inspect **myeloid signal in T cells and macrophages**, **TCR signal in T cells**, and **epithelial signal in tumor cells** together. Units are signal per 10,000 selected-model-gene UMIs for observed expression and the corresponding posterior expectation. The near–far contrast uses 25 µm and 50 µm distances to annotated macrophages. Check cell counts in each stratum before interpreting differences.

In [ ]:
display(pd.read_csv(out / "embedding_comparison.csv").round(3))
display(pd.read_csv(out / "specificity_and_distance.csv").round(3))
display(pd.read_csv(out / "marker_controls.csv").query("panel == 'myeloid' and compartment == 'T_panT_observed'"))
display(Image(filename=str(out / "marker_controls.png"), width=870))
display(Image(filename=str(out / "spatial_leiden.png"), width=980))

A useful pilot would show reduced myeloid signal in T cells alongside preserved macrophage marker expression and T-cell TCR expression, with improvement beyond nonspatial scVI. Spatial compartment or cell-state biology can also create a near–far gradient. Mixture fractions are model estimates. Set `MODE="full"` to repeat in one entire mouse with the same control panels and summaries. Only the 09B aggregate review ZIP is needed to share results. Local latent arrays follow the row order in `local_cell_assignments.csv.gz`.